In [23]:
# importing dependencies

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBRegressor
import joblib

In [6]:
# imporing the cleaned dataset which has already been analysed.

data = pd.read_excel(r'Data/cleaned_premiums.xlsx')

In [8]:
data.head()

,age,gender,region,marital_status,number_of_dependants,bmi_category,smoking_status,employment_status,income_lakhs,insurance_plan,genetical_risk,annual_premium_amount,medical_risk_score
0,26,Male,North,Unmarried,0,Normal,No Smoking,Salaried,6,Bronze,0,9053,6
1,29,Female,South,Married,2,Obesity,Regular,Salaried,6,Bronze,0,16339,6
2,49,Female,East,Married,2,Normal,No Smoking,Self-Employed,20,Silver,0,18164,6
3,30,Female,South,Married,3,Normal,No Smoking,Salaried,77,Gold,0,20303,0
4,56,Male,East,Married,3,Obesity,Occasional,Self-Employed,14,Bronze,0,15610,6


#### Splitting Data as per Age

In [9]:
print(data.shape[0])

df_young = data[data['age'] < 26]
df_old = data[data['age'] >= 26]

print(f'Length of df_young: {len(df_young)}, Length of df_old: {len(df_old)}')

49919
Length of df_young: 20089, Length of df_old: 29830


In [15]:
data['bmi_category'].value_counts()

bmi_category
Normal         23474
Overweight     11542
Underweight     7756
Obesity         7147
Name: count, dtype: int64

#### Making ML Pipeline for data preprocessing & model traning

In [28]:
def train_xgb_pipeline(data):
    X = data.drop(columns=['annual_premium_amount'])
    y = data['annual_premium_amount']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    #ordinal columns
    ordinal_cols = ['insurance_plan', 'bmi_category']
    ordinal_cat = [['Bronze', 'Silver', 'Gold'], ['Underweight', 'Normal', 'Overweight', 'Obesity']]

    #onehot columns
    onehot_cols = ['gender', 'marital_status', 'region','smoking_status','employment_status']

    #numeric columns
    numeric_cols = ['age', 'income_lakhs', 'number_of_dependants','genetical_risk','medical_risk_score']

    # Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('ord', OrdinalEncoder(categories=ordinal_cat), ordinal_cols),
            ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), onehot_cols),
            ('num', 'passthrough', numeric_cols)
        ]
    )

    # Pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(random_state=42))  
    ])

    param_dist = {
        'model__n_estimators': [100, 200, 300, 500],
        'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'model__max_depth': [3, 4, 5, 6, 8],
        'model__subsample': [0.7, 0.8, 0.9, 1.0],
        'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'model__min_child_weight': [1, 3, 5, 7]
    }

    # Train the model
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=20,
        scoring='neg_mean_absolute_error',
        cv=5,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_

    y_pred = best_model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2  :", r2)

    return best_model


#### Training models for both age groups

In [31]:
model_young = train_xgb_pipeline(df_young)
model_old = train_xgb_pipeline(df_old)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
MAE : 254.09983825683594
MSE : 86048.828125
RMSE: 293.34080542093017
R2  : 0.9888662695884705
Fitting 5 folds for each of 20 candidates, totalling 100 fits
MAE : 255.42889404296875
MSE : 89929.5234375
RMSE: 299.88251605837246
R2  : 0.9981632828712463


#### Exporting both models for production

In [32]:
joblib.dump(model_young, 'model_young.pkl')
joblib.dump(model_old, 'model_old.pkl')

['model_old.pkl']